# RI-JK + DFT UKS Hessian 分解概览 (TPSS0)

本文档对标 `06-1-decomp_nh3_r_tpss0.ipynb`。在 RKS 已经分解清楚的基础上，对 UKS (unrestricted KS) TPSS0 这一 hybrid meta-GGA 泛函进行 Hessian 分解。

相比 RKS 的主要差别：

1. UKS 使用 alpha/beta 两套自旋密度矩阵 `dm0a` 和 `dm0b`，总密度 `dm0 = dm0a + dm0b`。
2. 交换部分的系数不再是 0.5 * hyb，而是直接 hyb（因为每个自旋的 K 是独立的）。
3. DFT XC 部分的 `_get_vxc_diag` 和 `_get_vxc_deriv2` 返回 `(vmata, vmatb)` 对，分别对应 alpha 和 beta 自旋。
4. XC 核 `fxc` 的形状为 `[nvar, 2, nvar, 2, ngrids]`，即需要考虑自旋耦合。
5. CPHF 因子为 `2, 2, 1`（而非 RKS 的 `4, 4, 2`）。

In [1]:
from pyscf import gto, dft, lib
import numpy as np
from pyscf.hessian import rhf as rhf_hess
from pyscf.hessian import uks as uks_hess
from pyscf.df.hessian import uhf as df_uhf_hess
from functools import partial

lib.num_threads(16)
np.set_printoptions(5, suppress=True, linewidth=150)
np.einsum = partial(np.einsum, optimize="greedy")

In [2]:
xyz = """
N  0   0   0
H  1.0 0.1 0.2
H  0.3 1.1 0.2
H  0.1 0.1 1.2
"""

mol = gto.Mole(atom=xyz, basis="def2-TZVP", spin=2, charge=2, max_memory=32000).build()

In [3]:
mf = dft.UKS(mol, xc="TPSS0").density_fit()
dat0 = np.load("nh3_u_tpss0.npz")
mf.mo_coeff = dat0["mo_coeff"]
mf.mo_occ = dat0["mo_occ"]
mf.mo_energy = dat0["mo_energy"]
mf.with_df.build()
mf.converged = True

In [4]:
mf_hess = mf.Hessian()
mf_hess.auxbasis_response = 2
de_ref = mf_hess.kernel().copy()
print("de_ref shape:", de_ref.shape)
print("fp:", lib.fp(de_ref))

assert np.allclose(de_ref, dat0["ref_de"], rtol=1e-4, atol=1e-6)
print("de_ref matches reference: OK")


WARN: MGGA Hessian is sensitive to dft grids. grids.level 3 may not be dense enough.



de_ref shape: (4, 4, 3, 3)
fp: 0.5744843919375058
de_ref matches reference: OK


## Hybrid 系数与 XC 类型确认

TPSS0 是 hybrid meta-GGA，hybrid 系数 $c_K = 0.25$，且无 range separation、无 NLC。

In [5]:
ni = mf._numint
omega, alpha, hyb = ni.rsh_and_hybrid_coeff(mf.xc, spin=mol.spin)
print(f"omega={omega}, alpha={alpha}, hyb={hyb}")
print("is_hybrid_xc:", ni.libxc.is_hybrid_xc(mf.xc))
print("xc_type:", ni._xc_type(mf.xc))
print("do_nlc:", mf.do_nlc())
assert omega == 0.0  # no range separation
assert not mf.do_nlc()

omega=0.0, alpha=0.25, hyb=0.25
is_hybrid_xc: True
xc_type: MGGA
do_nlc: 0


## Hessian 分解概览

对于 UKS TPSS0 的 Hessian，其大致分为：

1. **密度矩阵非依赖项**：原子核排斥能的导数 `de_nuc`。
2. **密度矩阵一阶项导数**：`de_1`（hcore + ovlp 部分），由 `df_uhf_hess._partial_hess_ejk` 提供。
3. **J/K 复杂 skeleton 导数**：`de_J20/J11/J02` 和 `de_K20/K11/K02`。K 的最终系数是 $-c_K = -0.25$（注意 UKS 中没有 0.5 因子）。
4. **XC 数值积分 skeleton 导数**：`de_vxc`（新增）。由 `uks_hess._get_vxc_diag` 和 `uks_hess._get_vxc_deriv2` 组装。注意需要分别对 alpha 和 beta 密度矩阵进行收缩。
5. **CP-KS 贡献**：`de_cphf`。

In [6]:
# 1. Density matrix independent term (nuc-nuc only depends on mol, not spin)
de_nuc = rhf_hess.hess_nuc(mol)

In [7]:
# 2 + 3. Hcore / overlap skeleton; J/K with aux basis response
hessobj_aux0 = mf.Hessian()
hessobj_aux0.auxbasis_response = 0
de_1, ej_aux0, ek_aux0 = df_uhf_hess._partial_hess_ejk(hessobj_aux0, with_k=True)

In [8]:
hessobj_aux1 = mf.Hessian()
hessobj_aux1.auxbasis_response = 1
_, ej_aux1, ek_aux1 = df_uhf_hess._partial_hess_ejk(hessobj_aux1, with_k=True)

In [9]:
hessobj_aux2 = mf.Hessian()
hessobj_aux2.auxbasis_response = 2
_, ej_aux2, ek_aux2 = df_uhf_hess._partial_hess_ejk(hessobj_aux2, with_k=True)

In [10]:
# J/K skeleton decomposition
de_J20 = ej_aux0.copy()
de_J11 = 2.0 * (ej_aux1 - ej_aux0)
de_J02 = ej_aux2 - 2.0 * ej_aux1 + ej_aux0

# For UKS, the total formula is -hyb * (de_K20 + de_K11 + de_K02)
# (no 0.5 factor, unlike RKS where it's -0.5 * hyb)
# So we store the raw K pieces without the factor 2:
#   de_Ksum = de_K20 + de_K11 + de_K02 = ek_aux2
#   -hyb * de_Ksum = -hyb * ek_aux2  (matches PySCF partial hess)
de_K20 = ek_aux0.copy()
de_K11 = 2.0 * (ek_aux1 - ek_aux0)
de_K02 = ek_aux2 - 2.0 * ek_aux1 + ek_aux0

### 4-b. XC 数值积分 skeleton 二阶导数

对于 UKS 体系，`_get_vxc_diag` 返回 `(vmata, vmatb)`，`_get_vxc_deriv2` 返回 `(vxca, vxcb)` 列表对。组装时需要分别与 `dm0a` 和 `dm0b` 进行收缩，并乘以因子 2（对 +c.c.）。

In [11]:
# 4-b. XC numerical-integration skeleton (no grid response).
max_memory = 4000
veffa_diag, veffb_diag = uks_hess._get_vxc_diag(hessobj_aux0, mf.mo_coeff, mf.mo_occ, max_memory)
vxca, vxcb = uks_hess._get_vxc_deriv2(hessobj_aux0, mf.mo_coeff, mf.mo_occ, max_memory)

mocca = mf.mo_coeff[0][:, mf.mo_occ[0] > 0]
moccb = mf.mo_coeff[1][:, mf.mo_occ[1] > 0]
dm0a = mocca @ mocca.T
dm0b = moccb @ moccb.T
aoslices = mol.aoslice_by_atom()
natm = mol.natm
de_vxc = np.zeros((natm, natm, 3, 3))
for A in range(natm):
    p0, p1 = aoslices[A][2:]
    # Diagonal blocks: alpha + beta, factor 2 for +c.c.
    de_vxc[A, A] += np.einsum("xypq,pq->xy", veffa_diag[:, :, p0:p1], dm0a[p0:p1]) * 2
    de_vxc[A, A] += np.einsum("xypq,pq->xy", veffb_diag[:, :, p0:p1], dm0b[p0:p1]) * 2
    veffa_A = vxca[A]
    veffb_A = vxcb[A]
    for B in range(A + 1):
        q0, q1 = aoslices[B][2:]
        de_vxc[A, B] += np.einsum("xypq,pq->xy", veffa_A[:, :, q0:q1], dm0a[q0:q1]) * 2
        de_vxc[A, B] += np.einsum("xypq,pq->xy", veffb_A[:, :, q0:q1], dm0b[q0:q1]) * 2
    for B in range(A):
        de_vxc[B, A] = de_vxc[A, B].T

## CP-KS 响应

记录电子部分总贡献后，再减去已经分解出的电子项，剩下的就是 CP-KS 贡献。

In [12]:
# 5. CP-KS contribution
de_hess_elec = mf_hess.hess_elec()
de_partial = de_1 + ej_aux2 - hyb * ek_aux2 + de_vxc
de_cphf = de_hess_elec - de_partial


WARN: MGGA Hessian is sensitive to dft grids. grids.level 3 may not be dense enough.



## 总核验

总和形式为

$$
\mathrm{d}^2 E = \underbrace{\texttt{de\_1}}_{\text{hcore+ovlp}}
+ \underbrace{\texttt{de\_J20+de\_J11+de\_J02}}_{\text{J skeleton}}
- c_K \underbrace{(\texttt{de\_K20+de\_K11+de\_K02})}_{\text{K skeleton}}
+ \underbrace{\texttt{de\_vxc}}_{\text{XC skeleton}}
+ \underbrace{\texttt{de\_cphf}}_{\text{CP-KS}}
+ \underbrace{\texttt{de\_nuc}}_{\text{nuc-nuc}}
$$

其中 $c_K = 0.25$ 是 TPSS0 的 hybrid 系数。注意 UKS 中 K 的系数是 $-c_K$（不是 $-\frac{1}{2} c_K$）。

In [13]:
de_sum = (
    de_1
    + de_J20 + de_J11 + de_J02
    - hyb * (de_K20 + de_K11 + de_K02)
    + de_vxc
    + de_cphf
    + de_nuc
)
assert np.allclose(de_sum, de_ref, rtol=1e-4, atol=1e-6)
print("max abs difference:", np.max(np.abs(de_ref - de_sum)))

max abs difference: 1.511235581119763e-12


In [14]:
# --- beyond decomposition: XC intermediates and vmat_deriv1_mo ---
# Needed for Rust test validation; self-contained (defines its own mocca/moccb)
import sys
sys.path.insert(0, '..')
from pyhessref.nimatmul.uks import make_hessian_setup_batch_uks
from pyhessref.util import get_dm0_unrestricted

mocca = mf.mo_coeff[0][:, mf.mo_occ[0] > 0]
moccb = mf.mo_coeff[1][:, mf.mo_occ[1] > 0]
dm0_per_spin = get_dm0_unrestricted(mf.mo_coeff, mf.mo_occ)
dm0a_full, dm0b_full = dm0_per_spin[0], dm0_per_spin[1]

grids = dft.Grids(mol)
grids.coords = dat0["grid_coords"]
grids.weights = dat0["grid_weights"]

result = make_hessian_setup_batch_uks(
    mol, mf.xc, grids.coords, grids.weights, dm0a_full, dm0b_full
)

Ca = mf.mo_coeff[0]
Cb = mf.mo_coeff[1]
vmat_deriv1_mo_a = np.ascontiguousarray(
    np.einsum('pi,tapq,qj->taij', Ca, result['vmat_deriv1_a'], mocca)
)
vmat_deriv1_mo_b = np.ascontiguousarray(
    np.einsum('pi,tapq,qj->taij', Cb, result['vmat_deriv1_b'], moccb)
)
print(f"vmat_deriv1_mo_a shape: {vmat_deriv1_mo_a.shape}")
print(f"vmat_deriv1_mo_b shape: {vmat_deriv1_mo_b.shape}")
print(f"de_fxc fp:        {lib.fp(result['de_fxc']):.12f}")
print(f"vmat_deriv1_a fp: {lib.fp(result['vmat_deriv1_a']):.12f}")
print(f"vmat_deriv1_b fp: {lib.fp(result['vmat_deriv1_b']):.12f}")

Time for ao, rho, vxc, fxc: 0.209 s


Time for drho, de_fxc: 0.290 s
Time for de_vxc_diag: 0.118 s


Time for de_vxc_off: 0.187 s
Time for vmat_ip: 0.142 s


Time for vmat_deriv1: 0.496 s
vmat_deriv1_mo_a shape: (4, 3, 49, 5)
vmat_deriv1_mo_b shape: (4, 3, 49, 3)
de_fxc fp:        -28.913589693130
vmat_deriv1_a fp: -3.004579714990
vmat_deriv1_b fp: -2.792207211325


最终，我们将这些分量都放到 `nh3_u_tpss0_decomp.npz` 文件中，用于后续的核验和分析。

In [15]:
dat = dict(np.load("nh3_u_tpss0.npz"))
dat.update({
    "de_nuc": de_nuc,
    "de_1": de_1,
    "de_J20": de_J20,
    "de_J11": de_J11,
    "de_J02": de_J02,
    "de_K20": de_K20,
    "de_K11": de_K11,
    "de_K02": de_K02,
    "de_vxc": de_vxc,
    "de_cphf": de_cphf,
    "de_ref": de_ref,
    "hyb": np.asarray(hyb),
    # XC intermediate quantities (needed for Rust test validation)
    "de_fxc": np.ascontiguousarray(result["de_fxc"]),
    "de_vxc_diag_a": np.ascontiguousarray(result["de_vxc_diag_a"]),
    "de_vxc_diag_b": np.ascontiguousarray(result["de_vxc_diag_b"]),
    "de_vxc_off_a": np.ascontiguousarray(result["de_vxc_off_a"]),
    "de_vxc_off_b": np.ascontiguousarray(result["de_vxc_off_b"]),
    "vmat_ip_a": np.ascontiguousarray(result["vmat_ip_a"]),
    "vmat_ip_b": np.ascontiguousarray(result["vmat_ip_b"]),
    "vmat_deriv1_a": np.ascontiguousarray(result["vmat_deriv1_a"]),
    "vmat_deriv1_b": np.ascontiguousarray(result["vmat_deriv1_b"]),
    "vmat_deriv1_mo_a": vmat_deriv1_mo_a,
    "vmat_deriv1_mo_b": vmat_deriv1_mo_b,
})
np.savez("nh3_u_tpss0_decomp.npz", **dat)
print("Saved nh3_u_tpss0_decomp.npz")

Saved nh3_u_tpss0_decomp.npz
